# Analysis of Covid-19 papers

**Master:** Physics of Data \
**Course:** Management and Analysis of Physics Dataset - Module B (MAPD-B) \
**Professor:** Jacopo Pazzini \
**Students:** Polina Gridiaeva, Andres David Rojas Lozano, Elena Niero, Gabriela Landinez Rangel

## Task 0 - Downloading and Cleaning the data

This project concerns data from the real-world research on COVID-19 known as the COVID-19 Open Research Dataset Challenge (CORD-19). Information about the research and associated challenges are available on Kaggle: https://www.kaggle.com/allen-institute-for-ai/CORD-19-research-challenge. To download it we use:

```python
import kagglehub

# Download latest version
path = kagglehub.dataset_download("allen-institute-for-ai/CORD-19-research-challenge")

print("Path to dataset files:", path)
```

In the following notebook we perform the cleaning of the entire dataset from duplicated papers with the same titles.

### Spark set up

In [1]:
# Import the python libraries to create/connect to a Spark Session
from pyspark.sql import SparkSession

# Define the number of executors and number of cores for each one
executor_instances = 3
executor_cores = 4

# Build the spark session
spark = SparkSession.builder \
    .master("spark://master:7077") \
    .appName("Task 0 cleaning") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-common:3.3.4,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
    .config("spark.sql.execution.arrow.pyspark.fallback.enabled", "false")\
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')\
    .config("spark.hadoop.fs.s3a.access.key", "ACCESS_KEY") \
    .config("spark.hadoop.fs.s3a.secret.key", "SECRET_KEY") \
    .config('spark.hadoop.fs.s3a.endpoint', 'https://cloud-areapd.pd.infn.it:5210')\
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.metadatastore.impl", "org.apache.hadoop.fs.s3a.s3guard.NullMetadataStore") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled","false") \
    .config("com.amazonaws.sdk.disableCertChecking","true") \
    .config("spark.executor.instances", executor_instances) \
    .config("spark.executor.cores", executor_cores) \
    .config("spark.cores.max", executor_instances * executor_cores) \
    .config("spark.executor.memory", "10g") \
    .getOrCreate()

# Create a spark context and print its status
sc = spark.sparkContext
sc

:: loading settings :: url = jar:file:/mnt/vol2/env/jup_env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ubuntu/.ivy2.5.2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2.5.2/jars
org.apache.hadoop#hadoop-common added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2adf9888-afa1-4b99-958d-d464fbc37a4f;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-common;3.3.4 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-protobuf_3_7;1.1.1 in central
	found org.apache.hadoop#hadoop-annotations;3.3.4 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.1.1 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.cod

### Import all the other libraries

In [3]:
import os
import json
import re
import socket
import nltk
import pyspark.sql.functions as sf
import time
import pandas as pd
from pathlib import Path
from collections import Counter
from pyspark.sql.types import StructType, StructField, StringType

[nltk_data] Downloading package stopwords to /home/ubuntu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
nltk.download('stopwords')

In [4]:
%%time

base_dir = Path("/mnt/vol2/.cache/kagglehub/datasets/allen-institute-for-ai/CORD-19-research-challenge/versions/50/document_parses/")

all_paths = list(base_dir.rglob("*.json"))

print(f"Total JSON files found: {len(all_paths)}")
print(f"Total files found: {len(all_paths)}")

Total JSON files found: 183369
Total files found: 183369
CPU times: user 1.14 s, sys: 194 ms, total: 1.34 s
Wall time: 1.34 s


In [5]:
def paper_extraction(path):
    try:
        with open(path) as f:
            doc =json.load(f)
    except Exception:
        return []
    path_paper=path.name # This will save the address and extension of the file
    paper_id=doc.get("paper_id",None)
    meta = doc.get("metadata", {})
    title= meta.get("title",None)

    return [(path_paper, paper_id, title)]

In [6]:
%%time
paths_rdd = sc.parallelize(all_paths, numSlices=48)

schema = StructType([
    StructField("path", StringType(), True),
    StructField("paper_id", StringType(), True),
    StructField("title", StringType(), True)
])

paper_rdd = paths_rdd.flatMap(paper_extraction)
df_spark = spark.createDataFrame(paper_rdd, schema=schema) 

As the dataset is not that big , less than 1M rows, we can work with it as a Pandas dataframe easily.


In [7]:
%%time
columns = ['path', 'paper_id', 'title']
df_pandas = pd.DataFrame(paper_rdd.collect(), columns=columns)

In [10]:
df_pandas['title']=df_pandas['title'].str.upper()

In [11]:
df_pandas.head(100)

,path,paper_id,title
0,3703093f16e819c54aba603f7d7062d8a23c0d0c.json,3703093f16e819c54aba603f7d7062d8a23c0d0c,A SIGNAL-FLEXIBLE GENE DIAGNOSTIC STRATEGY COU...
1,fd738b997979c4b38c3a9aaa1b148a090fe94613.json,fd738b997979c4b38c3a9aaa1b148a090fe94613,"SUPPLEMENTARY MATERIALS: DESIGN, SYNTHESIS AND..."
2,09ce9181b6691229a1eef254eeb53d81c5361c6a.json,09ce9181b6691229a1eef254eeb53d81c5361c6a,SEVERITY PREDICTION FOR COVID-19 PATIENTS VIA ...
3,4bdf1cacda23b062421d549c0344a46c4afaabbc.json,4bdf1cacda23b062421d549c0344a46c4afaabbc,GENERATING FAKE BUT REALISTIC HEADLINES USING ...
4,49f9ed50153c6a24f1f483ee9d8dab6cfbb61f12.json,49f9ed50153c6a24f1f483ee9d8dab6cfbb61f12,WHEN NUCLEAR MEDICINE RADIOLOGICAL PROTECTION ...
...,...,...,...
95,c47d12cd6f0f28ee0401a4b83a39aaf9438ed15d.json,c47d12cd6f0f28ee0401a4b83a39aaf9438ed15d,EPIDEMIOLOGY AND INFECTION FROM THE FIELD
96,9241f46ecc8b63186b6e92d324dc8ad5266ba439.json,9241f46ecc8b63186b6e92d324dc8ad5266ba439,CHALLENGES OF ORAL MEDICINE SPECIALISTS AT THE...
97,7d37b9bb45aa60ff71b01a27afee24eca35597d4.json,7d37b9bb45aa60ff71b01a27afee24eca35597d4,TITLE: INTEGRATED DISEASE SURVEILLANCE AND RES...
98,1d2469d750e695530329e2fd064117ff984495de.json,1d2469d750e695530329e2fd064117ff984495de,ICU CAPACITY MANAGEMENT DURING THE COVID-19 PA...


In [12]:
df_pandas['title'].isna().sum()

np.int64(0)

In [13]:
total_duplicados = df_pandas['title'].duplicated().sum()

In [18]:
df_clean=df_pandas.drop_duplicates(subset=["title"])
df_clean.shape

(128782, 3)

In [19]:
df_clean.to_csv('/mnt/vol2/.cache/kagglehub/directory.csv',index=False)

/mnt/vol2/.cache/kagglehub/directory.csv


In [20]:
spark.stop()